[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module3_DeepLearning/14_NonlinearModelComparison.ipynb#copy=true)


# Choosing Nonlinear Models: Evaluation and Comparison

## Learning objectives

By the end of this lesson, you should be able to:

- compare a linear classifier with tree-based nonlinear models;
- use cross-validation to estimate performance during model selection;
- distinguish model selection from final test evaluation;
- communicate a model choice using accuracy, variability, and interpretability.

**Student Learning Outcome:** Select and justify a model using evidence from a reproducible evaluation workflow.


## A model score is not a model choice

Decision trees, random forests, and gradient boosting can represent nonlinear boundaries. Their apparent performance depends on the data split, hyperparameters, and metric. We will compare them with a regularized logistic-regression baseline using cross-validation on training data. Only after choosing a candidate should we consult the held-out test set.

Accuracy is appropriate here because the three wine classes are reasonably balanced. Other settings may require metrics that account for class imbalance or unequal error costs.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

RANDOM_STATE = 232
wine = load_wine(as_frame=True)
X, y = wine.data, wine.target
class_names = wine.target_names
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
print("features:", list(X.columns))
print("training observations:", len(X_train), "test observations:", len(X_test))


## Part 1 — Define comparable candidates

Logistic regression benefits from scaled numerical features. Tree-based models do not require scaling for their split rules. A `Pipeline` keeps scaling inside each cross-validation fit, preventing information from a validation fold from affecting its training transform.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

models = {
    "logistic regression": Pipeline([("scale", StandardScaler()),
                                     ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))]),
    "decision tree": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
    "random forest": RandomForestClassifier(n_estimators=300, max_features="sqrt",
                                              random_state=RANDOM_STATE, n_jobs=-1),
    "gradient boosting": HistGradientBoostingClassifier(learning_rate=0.08, max_iter=150,
                                                          max_leaf_nodes=12, l2_regularization=0.5,
                                                          random_state=RANDOM_STATE),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


## Part 2 — Cross-validate on training data

**Stratified** folds preserve class proportions approximately in every fold. Each candidate is fit five times, each time holding out a different training fold. We summarize mean and standard deviation of validation accuracy; the test set is still untouched.


In [ ]:
rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
    rows.append({"model": name, "mean validation accuracy": scores["test_score"].mean(),
                 "validation SD": scores["test_score"].std(),
                 "mean training accuracy": scores["train_score"].mean()})
cv_results = pd.DataFrame(rows).sort_values("mean validation accuracy", ascending=False)
cv_results.round(3)


### Practice 2

Which candidate would you investigate further? Justify your choice with mean validation accuracy and one non-score consideration, such as interpretability, training cost, or stability across folds.

> **YOUR ANSWER:**


## Part 3 — One final test evaluation

For demonstration, select the model with the largest mean validation accuracy, refit it on all training data, and evaluate it once on the held-out test set. In a real report, model choices must be fixed before this final evaluation.


In [ ]:
chosen_name = cv_results.iloc[0]["model"]
chosen_model = models[chosen_name].fit(X_train, y_train)
print("chosen model:", chosen_name)
print("final test accuracy:", round(chosen_model.score(X_test, y_test), 3))


### Checkpoint 3

Why would reporting the best test accuracy among all four models be misleading as an unbiased final result? Describe the role cross-validation played in avoiding this problem.

> **YOUR ANSWER:**


## Bridge to deep learning

Tree ensembles learn nonlinear structure by partitioning feature space. Neural networks learn nonlinear structure by composing weighted linear transformations with activation functions and optimizing parameters with gradients. The next lessons introduce PyTorch, the tensor and automatic-differentiation library used to build those networks.


## Lesson summary

There is no universally best nonlinear model. A credible comparison keeps a final test set separate, uses cross-validation for model selection, and weighs performance against interpretability, computation, and consequences of errors.
